# 🛠️ Building Custom Tools

**Day 10 — Notebook 2 of 5 | Estimated Time: 30 minutes**

---

## What You'll Learn
- What makes a good tool (and what makes a bad one)
- Building a web search simulation tool
- Database lookup and SQL query tools
- A generic HTTP API wrapper tool
- Tool error handling and graceful degradation
- Composing a multi-tool agent toolkit

---

## Principles of Good Tool Design

```
✅ Good tool                    ❌ Bad tool
──────────────────────────────────────────────────────
Single, clear purpose          Does too many things
Descriptive name               Vague name ("process")
Precise parameter descriptions Ambiguous param names
Returns structured data        Returns raw strings/HTML
Returns errors as data         Raises exceptions
Idempotent when possible       Side effects everywhere
```

---

## Setup

In [ ]:
%pip install -U -q "google-genai>=1.0.0"

In [ ]:
import sys, os, json, sqlite3, datetime, random

repo_root = os.path.abspath(os.path.join(os.getcwd(), "../../.."))
if repo_root not in sys.path:
    sys.path.append(repo_root)

from helpers.auth import get_api_key
from google import genai
from google.genai import types

client = genai.Client(api_key=get_api_key())
MODEL = "gemini-2.5-flash"
print("✅ Ready!")

---

## 1. The Agentic Loop Helper

We'll reuse the multi-step loop pattern from Notebook 1.

In [ ]:
def run_agent(query: str, tools: list, tool_registry: dict,
              system_prompt: str = "", max_steps: int = 8, verbose: bool = True) -> str:
    """
    Agentic loop: keep calling tools until Gemini produces a final text answer.
    """
    user_content = f"{system_prompt}\n\n{query}" if system_prompt else query
    contents = [types.Content(role="user", parts=[types.Part(text=user_content)])]

    for step in range(1, max_steps + 1):
        response = client.models.generate_content(
            model=MODEL, contents=contents,
            config=types.GenerateContentConfig(tools=tools),
        )
        part = response.candidates[0].content.parts[0]

        if not hasattr(part, "function_call") or part.function_call is None:
            return part.text.strip()

        fn_call = part.function_call
        fn_name, fn_args = fn_call.name, dict(fn_call.args)

        if verbose:
            print(f"  [step {step}] {fn_name}({json.dumps(fn_args, default=str)[:80]})")

        if fn_name not in tool_registry:
            fn_result = {"error": f"Tool '{fn_name}' not found in registry"}
        else:
            try:
                fn_result = tool_registry[fn_name](**fn_args)
            except Exception as e:
                fn_result = {"error": str(e)}

        if verbose:
            print(f"         → {str(fn_result)[:120]}")

        contents.append(types.Content(role="model", parts=[types.Part(function_call=fn_call)]))
        contents.append(types.Content(role="user", parts=[types.Part(
            function_response=types.FunctionResponse(name=fn_name, response=fn_result)
        )]))

    return "Max steps reached without a final answer."


print("Agent loop ready.")

---

## 2. Web Search Tool (Simulated)

In a real system you'd call a search API (SerpAPI, Brave Search, Tavily).  
Here we use a simulated local search over a news dataset so you can run offline.

In [ ]:
# Simulated news/knowledge articles
NEWS_DB = [
    {"title": "AI Investment Hits $91B in 2023", "date": "2024-01-10",
     "snippet": "Global AI investment reached $91 billion in 2023, led by generative AI companies. OpenAI raised $10B, Anthropic raised $7.3B. Enterprise AI adoption grew 35% year-on-year."},
    {"title": "Gemini 2.0 Released", "date": "2024-12-05",
     "snippet": "Google DeepMind released Gemini 2.0, featuring a 1M token context window, native tool use, and multimodal output capabilities including image and audio generation."},
    {"title": "Python Tops Developer Survey 2024", "date": "2024-06-15",
     "snippet": "Python remained the most used programming language for the fourth year running in the Stack Overflow Developer Survey 2024, with 51% of respondents using it regularly."},
    {"title": "UK AI Safety Institute Launches", "date": "2023-11-02",
     "snippet": "The UK government launched the AI Safety Institute to evaluate frontier AI risks. The institute focuses on evaluations for CBRN threats, societal harms, and autonomous behaviour."},
    {"title": "LangChain Reaches 100K GitHub Stars", "date": "2024-03-20",
     "snippet": "LangChain, the popular LLM framework, surpassed 100,000 GitHub stars in March 2024. The project introduced LangGraph for stateful multi-agent workflows."},
    {"title": "EU AI Act Signed Into Law", "date": "2024-07-12",
     "snippet": "The European Union AI Act was signed into law in July 2024, becoming the first comprehensive AI regulation globally. High-risk AI systems must comply by 2026."},
    {"title": "RAG Remains Dominant Enterprise AI Pattern", "date": "2024-09-08",
     "snippet": "A Gartner survey found RAG (Retrieval Augmented Generation) is the most widely deployed enterprise AI pattern, used in 73% of production LLM applications as of Q3 2024."},
]


def web_search(query: str, num_results: int = 3) -> dict:
    """Search local news database. Returns relevant articles."""
    query_lower = query.lower()
    scored = []
    for article in NEWS_DB:
        score = sum(
            word in article["title"].lower() or word in article["snippet"].lower()
            for word in query_lower.split()
        )
        if score > 0:
            scored.append((score, article))

    scored.sort(key=lambda x: x[0], reverse=True)
    results = [{"title": a["title"], "date": a["date"], "snippet": a["snippet"]}
               for _, a in scored[:num_results]]

    return {"results": results, "count": len(results)} if results else {"results": [], "message": "No results found"}


search_tool = types.Tool(
    function_declarations=[
        types.FunctionDeclaration(
            name="web_search",
            description="Search for recent news and information about AI, technology, and related topics.",
            parameters=types.Schema(
                type=types.Type.OBJECT,
                properties={
                    "query":       types.Schema(type=types.Type.STRING, description="The search query"),
                    "num_results": types.Schema(type=types.Type.INTEGER, description="Number of results to return (default 3, max 5)"),
                },
                required=["query"],
            ),
        )
    ]
)

# Test
print("Search test:")
result = web_search("AI investment 2024")
for r in result["results"]:
    print(f"  [{r['date']}] {r['title']}")

---

## 3. Database Query Tool

A safe read-only SQL tool over a local SQLite database.

In [ ]:
# Create a sample SQLite database
DB_PATH = "/tmp/company.db"

def setup_database():
    conn = sqlite3.connect(DB_PATH)
    c = conn.cursor()
    c.executescript("""
    DROP TABLE IF EXISTS employees;
    DROP TABLE IF EXISTS departments;
    DROP TABLE IF EXISTS projects;

    CREATE TABLE departments (
        id TEXT PRIMARY KEY, name TEXT, budget_gbp INTEGER, head_count INTEGER
    );
    CREATE TABLE employees (
        id TEXT PRIMARY KEY, name TEXT, department_id TEXT, role TEXT,
        salary_gbp INTEGER, start_date TEXT
    );
    CREATE TABLE projects (
        id TEXT PRIMARY KEY, name TEXT, department_id TEXT,
        status TEXT, budget_gbp INTEGER, deadline TEXT
    );

    INSERT INTO departments VALUES
        ('eng',     'Engineering',    2000000, 45),
        ('product', 'Product',         800000, 18),
        ('data',    'Data Science',   1200000, 22),
        ('hr',      'Human Resources', 400000, 8);

    INSERT INTO employees VALUES
        ('e01','Alice Chen',   'eng',     'Senior Engineer',      95000, '2021-03-15'),
        ('e02','Bob Patel',    'data',    'Data Scientist',        82000, '2022-07-01'),
        ('e03','Carol Smith',  'product', 'Product Manager',       88000, '2020-01-20'),
        ('e04','David Lee',    'eng',     'Staff Engineer',       115000, '2019-09-10'),
        ('e05','Eva Müller',   'data',    'ML Engineer',           90000, '2021-11-30'),
        ('e06','Frank Okafor', 'hr',      'HR Business Partner',   72000, '2023-02-14'),
        ('e07','Grace Kim',    'eng',     'Junior Engineer',       65000, '2023-08-01'),
        ('e08','Henry Zhao',   'product', 'UX Designer',          75000, '2022-04-05');

    INSERT INTO projects VALUES
        ('p01','AI Assistant v2',  'eng',     'active',    350000, '2025-06-30'),
        ('p02','Data Platform',    'data',    'active',    280000, '2025-03-31'),
        ('p03','Customer Portal',  'product', 'completed',  95000, '2024-12-15'),
        ('p04','HR Automation',    'hr',      'planning',   60000, '2025-09-01'),
        ('p05','Recommendation ML','data',    'active',    175000, '2025-05-15');
    """)
    conn.commit()
    conn.close()

setup_database()
print("✅ Database created at", DB_PATH)


ALLOWED_TABLES = {"employees", "departments", "projects"}

def query_database(sql: str) -> dict:
    """
    Execute a READ-ONLY SQL query on the company database.
    Only SELECT statements are allowed.
    """
    # Safety: only allow SELECT
    sql_clean = sql.strip()
    if not sql_clean.upper().startswith("SELECT"):
        return {"error": "Only SELECT queries are allowed"}

    try:
        conn = sqlite3.connect(DB_PATH)
        conn.row_factory = sqlite3.Row
        cursor = conn.execute(sql_clean)
        rows = [dict(row) for row in cursor.fetchall()]
        conn.close()
        return {"rows": rows, "count": len(rows)}
    except Exception as e:
        return {"error": str(e)}


def get_db_schema() -> dict:
    """Return the database table schemas."""
    schema = {}
    conn = sqlite3.connect(DB_PATH)
    for table in ALLOWED_TABLES:
        cursor = conn.execute(f"PRAGMA table_info({table})")
        schema[table] = [row[1] for row in cursor.fetchall()]
    conn.close()
    return {"schema": schema}


db_tool = types.Tool(
    function_declarations=[
        types.FunctionDeclaration(
            name="query_database",
            description="Query the company database with a SELECT SQL statement. Tables: employees (id, name, department_id, role, salary_gbp, start_date), departments (id, name, budget_gbp, head_count), projects (id, name, department_id, status, budget_gbp, deadline).",
            parameters=types.Schema(
                type=types.Type.OBJECT,
                properties={
                    "sql": types.Schema(type=types.Type.STRING, description="A valid SQL SELECT statement"),
                },
                required=["sql"],
            ),
        ),
        types.FunctionDeclaration(
            name="get_db_schema",
            description="Get the database schema — table names and column names.",
            parameters=types.Schema(type=types.Type.OBJECT, properties={}),
        ),
    ]
)

# Quick test
r = query_database("SELECT name, salary_gbp FROM employees ORDER BY salary_gbp DESC LIMIT 3")
print("Top earners:", r["rows"])

In [ ]:
# Test DB tool with the agent
db_registry = {"query_database": query_database, "get_db_schema": get_db_schema}

db_queries = [
    "Who are the employees in the Data Science department?",
    "What is the total salary budget for Engineering?",
    "Which projects are currently active and what are their budgets?",
]

for q in db_queries:
    print(f"\nQuery: {q}")
    answer = run_agent(q, [db_tool], db_registry)
    print(f"Answer: {answer}")

---

## 4. Date and Time Tools

LLMs don't know the current date. A simple datetime tool grounds them in reality.

In [ ]:
def get_current_datetime(timezone: str = "UTC") -> dict:
    """Return the current date and time."""
    now = datetime.datetime.utcnow()
    return {
        "datetime_utc": now.isoformat(),
        "date": now.strftime("%Y-%m-%d"),
        "time": now.strftime("%H:%M:%S"),
        "day_of_week": now.strftime("%A"),
        "week_number": now.isocalendar()[1],
    }


def calculate_date_diff(date1: str, date2: str) -> dict:
    """Calculate the number of days between two dates (YYYY-MM-DD format)."""
    try:
        d1 = datetime.date.fromisoformat(date1)
        d2 = datetime.date.fromisoformat(date2)
        diff = (d2 - d1).days
        return {
            "days": abs(diff),
            "weeks": abs(diff) // 7,
            "direction": "d2 is after d1" if diff >= 0 else "d1 is after d2",
        }
    except ValueError as e:
        return {"error": str(e)}


datetime_tool = types.Tool(
    function_declarations=[
        types.FunctionDeclaration(
            name="get_current_datetime",
            description="Get the current date and time in UTC. Use this whenever the user asks about today's date, the current time, or what day it is.",
            parameters=types.Schema(
                type=types.Type.OBJECT,
                properties={
                    "timezone": types.Schema(type=types.Type.STRING, description="Timezone name (default: UTC)"),
                },
            ),
        ),
        types.FunctionDeclaration(
            name="calculate_date_diff",
            description="Calculate the number of days between two dates.",
            parameters=types.Schema(
                type=types.Type.OBJECT,
                properties={
                    "date1": types.Schema(type=types.Type.STRING, description="First date in YYYY-MM-DD format"),
                    "date2": types.Schema(type=types.Type.STRING, description="Second date in YYYY-MM-DD format"),
                },
                required=["date1", "date2"],
            ),
        ),
    ]
)

dt_registry = {"get_current_datetime": get_current_datetime, "calculate_date_diff": calculate_date_diff}

print("Datetime tool test:")
answer = run_agent("What day of the week is it today, and how many days until Christmas (Dec 25)?",
                   [datetime_tool], dt_registry)
print(f"Answer: {answer}")

---

## 5. Composing a Multi-Tool Toolkit

A real agent has access to multiple tools and picks the right one per task.

In [ ]:
# Combine all tools into one toolkit
ALL_TOOLS = [search_tool, db_tool, datetime_tool]

ALL_REGISTRY = {
    "web_search": web_search,
    "query_database": query_database,
    "get_db_schema": get_db_schema,
    "get_current_datetime": get_current_datetime,
    "calculate_date_diff": calculate_date_diff,
}

SYSTEM_PROMPT = """You are a helpful assistant with access to company data, news search,
and date/time tools. Always use tools to get accurate information rather than relying 
on your training data for facts that could be time-sensitive."""

compound_queries = [
    "What is the total project budget for the Data Science department? "
    "And how many days until the nearest project deadline?",

    "What's the latest AI news, and how long ago was the EU AI Act signed?",
]

for q in compound_queries:
    print(f"\n{'='*65}")
    print(f"Query: {q}")
    print("-"*65)
    answer = run_agent(q, ALL_TOOLS, ALL_REGISTRY, system_prompt=SYSTEM_PROMPT)
    print(f"\nAnswer: {answer}")

---

## 6. Tool Error Handling

Tools will fail. Graceful error responses let the agent recover rather than crash.

In [ ]:
# Demonstrate error handling: bad SQL, unknown table
bad_queries = [
    "SELECT * FROM salaries",                # non-existent table
    "DELETE FROM employees WHERE id='e01'",  # write operation blocked
]

print("Error handling demo:\n")
for sql in bad_queries:
    result = query_database(sql)
    print(f"  SQL: {sql}")
    print(f"  Result: {result}\n")

# When the agent gets an error, it should handle gracefully
print("Agent behaviour on error:")
answer = run_agent(
    "Query the salaries table for all employees.",
    [db_tool], db_registry
)
print(f"Answer: {answer}")

---

## 🏋️ Exercise 1: Weather Tool

Build a simulated weather tool that returns structured weather data.

In [ ]:
# Exercise 1: Simulated weather tool
# TODO:
# 1. Create a get_weather(city: str, unit: str = "celsius") → dict function
#    Simulate weather data for at least 5 cities:
#    e.g. London, New York, Tokyo, Sydney, Paris
#    Return: {"city": ..., "temperature": ..., "condition": ..., "humidity": ..., "wind_kph": ...}
#    For unknown cities, return {"error": "City not found"}
#
# 2. Create the types.Tool declaration
#
# 3. Add to ALL_REGISTRY and ALL_TOOLS
#
# 4. Test with multi-tool queries like:
#    "What's the weather in London? And how does today's temperature compare to the seasonal average?"
#    "Which is warmer right now: Tokyo or Paris?"

CITY_WEATHER = {
    # TODO: Populate with weather data
    # "london": {"temperature_c": 12, "condition": "Overcast", "humidity": 78, "wind_kph": 15},
}

def get_weather(city: str, unit: str = "celsius") -> dict:
    # TODO: Implement
    pass

weather_tool = None  # TODO: Create types.Tool declaration

---

## 🏋️ Exercise 2: Tool Description Quality

The description is what Gemini reads to decide whether to use a tool. Bad descriptions cause bad routing.

In [ ]:
# Exercise 2: Description quality comparison
# Run the same query against two versions of the search tool:
#   - search_tool_vague: description = "Search for information"
#   - search_tool_precise: description = the full description we wrote above
#
# Query: "What has happened recently with AI regulation?"
#
# For each version:
#   1. Does Gemini call the tool?
#   2. Does it use the right parameters?
#
# TODO: Implement and compare

search_tool_vague = types.Tool(
    function_declarations=[
        types.FunctionDeclaration(
            name="web_search",
            description="Search for information.",  # intentionally vague
            parameters=types.Schema(
                type=types.Type.OBJECT,
                properties={
                    "query": types.Schema(type=types.Type.STRING, description="search query"),
                },
                required=["query"],
            ),
        )
    ]
)

test_query = "What has happened recently with AI regulation in Europe?"

print("Vague description:")
# TODO: run with search_tool_vague and print result

print("\nPrecise description:")
# TODO: run with search_tool (the original) and compare

---

## Key Takeaways

| Tool | Purpose | Key Design Choice |
|------|---------|-------------------|
| Web search | Real-time information retrieval | Returns structured results, not raw HTML |
| Database query | Read company/domain data | Whitelist SELECT only; return errors as dicts |
| Datetime | Ground the LLM in the present | Called whenever "today"/"now" appears |
| Multi-tool | Complex compound tasks | Agent loops through calls automatically |

**Tool description is the most important design decision** — it determines when Gemini calls the tool.

---

## 📚 Further Reading

| Resource | Type | Description |
|----------|------|-------------|
| [Tavily Search API](https://tavily.com/) | Tool | Real web search API designed for AI agents |
| [Tool Use Best Practices](https://ai.google.dev/gemini-api/docs/function-calling#best-practices) | Docs | Official Gemini tool design guidance |

---

**Next up:** [03_Introduction_to_Agents.ipynb](./03_Introduction_to_Agents.ipynb)